In [21]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline
from tensorflow import keras
import seaborn as sns 

In [22]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [23]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [24]:
train.drop(['id'], axis = 1, inplace = True)

In [25]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd

# Separate target
X = train.drop("PitNextLap", axis=1)
y = train["PitNextLap"].astype(int)

# Columns
categorical_cols = ["Driver", "Compound", "Race"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, y_train.shape

((351312, 929), (351312,))

In [27]:
test_ids = test["id"]

X_test = test.drop("id", axis=1)

X_test_processed = preprocessor.transform(X_test)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()

# Input layer + first hidden layer
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))

# Second hidden layer
model.add(Dense(68, activation='relu'))
model.add(Dropout(0.2))

# Output layer (binary classification)
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='relu'))

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # binary target
    metrics=['accuracy']
)

In [30]:
import tensorflow as tf

class PrintSelectedEpochs(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        epoch_num = epoch + 1

        if epoch_num == 1 or epoch_num % 50 == 0:
            print(
                f"Epoch {epoch_num}: "
                f"loss={logs['loss']:.4f}, "
                f"accuracy={logs['accuracy']:.4f}, "
                f"val_loss={logs['val_loss']:.4f}, "
                f"val_accuracy={logs['val_accuracy']:.4f}"
            )

In [31]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=128,
    verbose=0,
    callbacks=[PrintSelectedEpochs()]
)

Epoch 1: loss=0.3715, accuracy=0.8453, val_loss=0.3199, val_accuracy=0.8484
Epoch 50: loss=0.3278, accuracy=0.8855, val_loss=0.3157, val_accuracy=0.8863
Epoch 100: loss=0.3796, accuracy=0.8808, val_loss=0.5073, val_accuracy=0.8795
Epoch 150: loss=0.3880, accuracy=0.8840, val_loss=0.4075, val_accuracy=0.8809
Epoch 200: loss=0.4243, accuracy=0.8745, val_loss=0.4045, val_accuracy=0.8802


In [34]:
val_loss, val_acc = model.evaluate(X_val, y_val)
print("Validation Accuracy:", val_acc)

2745/2745 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8802 - loss: 0.4045
Validation Accuracy: 0.8801862597465515


In [36]:
features = categorical_cols + numeric_cols

X_competition = test[features]

X_competition_scaled = preprocessor.transform(X_competition)

y_pred_prob = model.predict(X_competition_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": y_pred.flatten()
})

submission.to_csv("submission.csv", index=False)

5881/5881 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step
